# Traffic Patterns Forecasting — Time Series Forecasting

**Project:** Traffic Patterns Forecasting with Time Series Models  
**Dataset:** PEMS-BAY traffic-speed benchmark  
**Forecast horizon:** Next 1 hour (12 × 5-minute steps)  
**Approach:** EDA → feature engineering → chronological split → Naive baseline → Random Forest → LSTM → evaluation → future forecast

> The notebook downloads the public PEMS-BAY CSV automatically. PEMS-BAY contains 325 traffic sensors sampled every 5 minutes and is widely used for traffic forecasting. The dataset is available through Zenodo and newer processed versions are also available through Hugging Face. 


In [ ]:
# 1. Install/import dependencies
# Run this cell if packages are missing:
# %pip install -q pandas numpy matplotlib seaborn scikit-learn tensorflow requests

import os
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

DATA_DIR = Path("traffic_data")
DATA_DIR.mkdir(exist_ok=True)

print("Environment ready.")


## 2. Download the latest available public benchmark data

This notebook uses the public PEMS-BAY benchmark archive. The commonly distributed CSV contains traffic measurements from 325 sensors at 5-minute intervals.

The dataset itself is historical rather than live. Therefore, “latest data” here means the latest publicly available version of the selected benchmark/archive, not real-time traffic in 2026.


In [ ]:
# Download PEMS-BAY from the public Zenodo archive if it is not already present.
import requests

csv_path = DATA_DIR / "PEMS-BAY.csv"
url = "https://zenodo.org/records/5146275/files/PEMS-BAY.csv?download=1"

if not csv_path.exists():
    print("Downloading PEMS-BAY.csv ...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    csv_path.write_bytes(r.content)
    print(f"Downloaded {csv_path.stat().st_size / 1024**2:.1f} MB")
else:
    print("Dataset already exists:", csv_path)

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
display(df.head())


In [ ]:
# 3. Inspect the dataset
print("Columns:", len(df.columns))
print("Rows:", len(df))
display(df.head())
display(df.dtypes.head(10))


In [ ]:
# 4. Convert the timestamp column and inspect missing values
timestamp_col = df.columns[0]
df[timestamp_col] = pd.to_datetime(df[timestamp_col])

df = df.sort_values(timestamp_col).set_index(timestamp_col)

print("Time range:", df.index.min(), "to", df.index.max())
print("Frequency estimate:", df.index.to_series().diff().mode().iloc[0])

missing = df.isna().mean().sort_values(ascending=False)
print("Sensors with missing values:", (missing > 0).sum())
display(missing.head(10))


## 5. Select a sensor

For a practical univariate time-series model, we forecast one sensor at a time. You can change `SENSOR_ID` below to another sensor column.

The complete dataset contains many sensors, so the same pipeline can later be extended to multivariate or graph-based forecasting.


In [ ]:
# Choose a sensor automatically, or replace with a specific column name.
SENSOR_ID = df.columns[0]

series = df[SENSOR_ID].copy().interpolate().ffill().bfill()

print("Selected sensor:", SENSOR_ID)
print("Observations:", len(series))
print(series.describe())


In [ ]:
# 6. Traffic pattern visualization
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(series.index, series.values, linewidth=0.8)
ax.set_title(f"Traffic Speed — Sensor {SENSOR_ID}")
ax.set_xlabel("Time")
ax.set_ylabel("Speed")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 7. Recent traffic pattern
recent = series.tail(7 * 24 * 12)  # approximately last 7 days at 5-minute intervals

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(recent.index, recent.values, linewidth=1)
ax.set_title(f"Recent 7-Day Traffic Pattern — Sensor {SENSOR_ID}")
ax.set_xlabel("Time")
ax.set_ylabel("Speed")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 8. Feature engineering
data = pd.DataFrame({"traffic": series})

data["hour"] = data.index.hour
data["minute"] = data.index.minute
data["dayofweek"] = data.index.dayofweek
data["dayofyear"] = data.index.dayofyear
data["is_weekend"] = (data["dayofweek"] >= 5).astype(int)

# Cyclic time features
data["hour_sin"] = np.sin(2 * np.pi * (data["hour"] + data["minute"]/60) / 24)
data["hour_cos"] = np.cos(2 * np.pi * (data["hour"] + data["minute"]/60) / 24)
data["dow_sin"] = np.sin(2 * np.pi * data["dayofweek"] / 7)
data["dow_cos"] = np.cos(2 * np.pi * data["dayofweek"] / 7)

# Lag features: 5 min, 15 min, 30 min, 1 hour, 2 hours, 1 day, 1 week
lags = [1, 3, 6, 12, 24, 288, 2016]
for lag in lags:
    data[f"lag_{lag}"] = data["traffic"].shift(lag)

# Rolling statistics
data["rolling_mean_1h"] = data["traffic"].shift(1).rolling(12).mean()
data["rolling_std_1h"] = data["traffic"].shift(1).rolling(12).std()
data["rolling_mean_3h"] = data["traffic"].shift(1).rolling(36).mean()

data = data.dropna()

print("Feature matrix:", data.shape)
display(data.head())


## 9. Chronological train/validation/test split

For time-series forecasting, random train/test splitting can leak future information. We therefore preserve chronological order:

- **70% training**
- **15% validation**
- **15% test**

The test set is used only for final evaluation.


In [ ]:
n = len(data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = data.iloc[:train_end]
val = data.iloc[train_end:val_end]
test = data.iloc[val_end:]

feature_cols = [c for c in data.columns if c != "traffic"]

X_train, y_train = train[feature_cols], train["traffic"]
X_val, y_val = val[feature_cols], val["traffic"]
X_test, y_test = test[feature_cols], test["traffic"]

print("Train:", train.index.min(), "→", train.index.max(), len(train))
print("Validation:", val.index.min(), "→", val.index.max(), len(val))
print("Test:", test.index.min(), "→", test.index.max(), len(test))


In [ ]:
# 10. Metrics helper
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")
    return {"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = []


## 11. Baseline: persistence forecast

The baseline predicts the current traffic value as the next value. This is important because a machine-learning model should demonstrate improvement over a simple forecasting rule.


In [ ]:
baseline_pred = X_test["lag_1"].values
results.append(evaluate_model("Persistence baseline", y_test.values, baseline_pred))


## 12. Random Forest time-series model

Random Forest is used as a strong non-neural baseline using lagged traffic and calendar features.

It does not inherently understand sequence order, so the lag and time features provide the temporal information explicitly.


In [ ]:
rf = RandomForestRegressor(
    n_estimators=250,
    max_depth=18,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
results.append(evaluate_model("Random Forest", y_test.values, rf_pred))


In [ ]:
# 13. Feature importance
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importance.head(15).sort_values().plot(kind="barh")
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

display(importance.head(15))


## 14. LSTM sequence model

The LSTM uses a rolling history window to learn temporal dependencies directly.

For a practical notebook, the sequence length is 288 points = 24 hours at 5-minute intervals.


In [ ]:
# TensorFlow/Keras LSTM
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow:", tf.__version__)

LOOKBACK = 288  # 24 hours
target = series.values.astype("float32")

# Scale using training data only
split_train = int(len(target) * 0.70)
split_val = int(len(target) * 0.85)

scaler = StandardScaler()
scaler.fit(target[:split_train].reshape(-1, 1))
scaled = scaler.transform(target.reshape(-1, 1)).flatten()

def make_sequences(values, start, end, lookback):
    X, y = [], []
    for i in range(max(start, lookback), end):
        X.append(values[i-lookback:i])
        y.append(values[i])
    return np.array(X)[..., None], np.array(y)

X_lstm_train, y_lstm_train = make_sequences(scaled, 0, split_train, LOOKBACK)
X_lstm_val, y_lstm_val = make_sequences(scaled, split_train, split_val, LOOKBACK)
X_lstm_test, y_lstm_test = make_sequences(scaled, split_val, len(scaled), LOOKBACK)

print(X_lstm_train.shape, X_lstm_val.shape, X_lstm_test.shape)


In [ ]:
# 15. Build and train LSTM
lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOKBACK, 1)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1)
])

lstm.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = lstm.fit(
    X_lstm_train,
    y_lstm_train,
    validation_data=(X_lstm_val, y_lstm_val),
    epochs=30,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
# 16. LSTM training curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history.history["loss"], label="Train loss")
ax.plot(history.history["val_loss"], label="Validation loss")
ax.set_title("LSTM Training History")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 17. LSTM test evaluation
lstm_scaled_pred = lstm.predict(X_lstm_test, verbose=0).flatten()

lstm_pred = scaler.inverse_transform(
    lstm_scaled_pred.reshape(-1, 1)
).flatten()

lstm_actual = scaler.inverse_transform(
    y_lstm_test.reshape(-1, 1)
).flatten()

results.append(evaluate_model("LSTM", lstm_actual, lstm_pred))


In [ ]:
# 18. Compare models
results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

plt.figure(figsize=(9, 5))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.title("Model RMSE Comparison")
plt.ylabel("RMSE")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# 19. Plot actual vs Random Forest predictions
plot_n = min(1000, len(y_test))

plt.figure(figsize=(15, 5))
plt.plot(y_test.index[-plot_n:], y_test.values[-plot_n:], label="Actual", linewidth=1.2)
plt.plot(y_test.index[-plot_n:], rf_pred[-plot_n:], label="Random Forest", linewidth=1)
plt.title(f"Actual vs Random Forest — Sensor {SENSOR_ID}")
plt.xlabel("Time")
plt.ylabel("Traffic speed")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
# 20. One-hour recursive LSTM forecast
# Forecasts the next 12 five-minute intervals from the latest available history.

history_values = scaled[-LOOKBACK:].copy()
future_scaled = []

for _ in range(12):
    x = history_values[-LOOKBACK:].reshape(1, LOOKBACK, 1)
    pred = lstm.predict(x, verbose=0)[0, 0]
    future_scaled.append(pred)
    history_values = np.append(history_values, pred)

future_forecast = scaler.inverse_transform(
    np.array(future_scaled).reshape(-1, 1)
).flatten()

future_index = pd.date_range(
    start=series.index[-1] + pd.Timedelta(minutes=5),
    periods=12,
    freq="5min"
)

forecast_df = pd.DataFrame({
    "timestamp": future_index,
    "forecast_speed": future_forecast
})

display(forecast_df)


In [ ]:
# 21. Visualize the next-hour forecast
history_plot = series.tail(144)  # last 12 hours

plt.figure(figsize=(15, 5))
plt.plot(history_plot.index, history_plot.values, label="Historical", linewidth=1.2)
plt.plot(
    forecast_df["timestamp"],
    forecast_df["forecast_speed"],
    marker="o",
    label="Next-hour forecast"
)
plt.axvline(series.index[-1], linestyle="--", alpha=0.7)
plt.title(f"Next 1-Hour Traffic Forecast — Sensor {SENSOR_ID}")
plt.xlabel("Time")
plt.ylabel("Forecast traffic speed")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 22. Traffic pattern analysis

The following section summarizes hourly and weekday patterns from the selected sensor. These patterns can be useful for identifying recurring congestion periods.


In [ ]:
pattern = pd.DataFrame({"traffic": series})
pattern["hour"] = pattern.index.hour
pattern["weekday"] = pattern.index.day_name()

hourly = pattern.groupby("hour")["traffic"].mean()
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday = pattern.groupby("weekday")["traffic"].mean().reindex(weekday_order)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hourly.index, hourly.values, marker="o")
ax.set_title("Average Traffic Speed by Hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Average speed")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(weekday.index, weekday.values)
ax.set_title("Average Traffic Speed by Day of Week")
ax.set_ylabel("Average speed")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

display(hourly.to_frame("mean_speed"))
display(weekday.to_frame("mean_speed"))


## 23. Conclusions

### What this project demonstrates

1. Traffic data is treated as a chronological time series.
2. Lag and calendar features capture recurring traffic patterns.
3. A persistence baseline establishes a simple benchmark.
4. Random Forest provides a non-neural forecasting model.
5. LSTM learns temporal dependencies from a 24-hour history window.
6. Models are evaluated using MAE, RMSE and R².
7. The final LSTM produces a 1-hour / 12-step traffic forecast.

### Important limitation

PEMS-BAY is a historical benchmark rather than a live traffic feed. For a real-world deployment, replace the download section with a current traffic API or sensor stream, then retrain/recalibrate the forecasting pipeline regularly.

### Possible extensions

- Multi-step direct forecasting instead of recursive LSTM forecasting
- XGBoost/LightGBM comparison
- GRU and Transformer models
- Weather and holiday features
- Multi-sensor forecasting
- Graph Neural Networks using the sensor adjacency matrix
- Prediction intervals / uncertainty forecasting
- Real-time dashboard with Streamlit


## References

- PEMS-BAY public dataset archive: Zenodo, DOI **10.5281/zenodo.4264445**
- Li et al., *Diffusion Convolutional Recurrent Neural Network: Data-Driven Traffic Forecasting*, ICLR 2018.
- Processed PEMS-BAY benchmark documentation: Hugging Face dataset card.

**Reproducibility:** Random Forest uses `random_state=42`; the time-series split is chronological to reduce temporal leakage.
